# PRODIGY_DS_03 — Decision Tree Classifier
## Prodigy InfoTech Data Science Internship | Task 3

**Objective:** Build a Decision Tree Classifier to predict whether a customer will subscribe to a term deposit based on their demographic and behavioral data.

**Dataset:** Bank Marketing Dataset (UCI Machine Learning Repository)

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay)
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

## 2. Load & Explore Dataset

In [ ]:
df = pd.read_csv('bank.csv', sep=';')
print("Dataset Shape:", df.shape)
df.head()

In [ ]:
# Basic Info
print("Dataset Info:")
df.info()

In [ ]:
# Missing values check
print("Missing Values:")
print(df.isnull().sum())
print("\nNo missing values found!" if df.isnull().sum().sum()==0 else "Missing values exist!")

In [ ]:
# Statistical Summary
df.describe()

In [ ]:
# Target Variable Distribution
print("Target Variable Distribution:")
print(df['y'].value_counts())
print(f"\nClass Imbalance Ratio: {df['y'].value_counts()['no']/df['y'].value_counts()['yes']:.2f}:1 (No:Yes)")

## 3. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Exploratory Data Analysis — Bank Marketing Dataset', fontsize=16, fontweight='bold')

# Age distribution
axes[0,0].hist(df['age'], bins=30, color='steelblue', edgecolor='white')
axes[0,0].set_title('Age Distribution')
axes[0,0].set_xlabel('Age'); axes[0,0].set_ylabel('Count')

# Target distribution
colors = ['#e74c3c','#2ecc71']
df['y'].value_counts().plot(kind='bar', ax=axes[0,1], color=colors, edgecolor='white', rot=0)
axes[0,1].set_title('Target Variable Distribution (y)')
axes[0,1].set_xlabel('Subscribed'); axes[0,1].set_ylabel('Count')
for p in axes[0,1].patches:
    axes[0,1].annotate(str(p.get_height()), (p.get_x()+0.1, p.get_height()+20))

# Job vs subscription rate
job_sub = df.groupby('job')['y'].apply(lambda x: (x=='yes').mean()*100).sort_values(ascending=False)
job_sub.plot(kind='bar', ax=axes[0,2], color='mediumpurple', edgecolor='white', rot=45)
axes[0,2].set_title('Subscription Rate by Job (%)')
axes[0,2].set_ylabel('Subscription Rate %')

# Duration vs outcome
df.boxplot(column='duration', by='y', ax=axes[1,0])
axes[1,0].set_title('Call Duration vs Subscription')
axes[1,0].set_xlabel('Subscribed'); axes[1,0].set_ylabel('Duration (seconds)')
plt.sca(axes[1,0]); plt.title('Call Duration vs Subscription')

# Balance by outcome
df[df['y']=='yes']['balance'].hist(ax=axes[1,1], bins=40, alpha=0.6, color='green', label='Yes')
df[df['y']=='no']['balance'].hist(ax=axes[1,1], bins=40, alpha=0.6, color='red', label='No')
axes[1,1].set_title('Balance Distribution by Subscription')
axes[1,1].set_xlabel('Balance (€)'); axes[1,1].legend()
axes[1,1].set_xlim(-3000, 15000)

# Education vs subscription
edu_sub = df.groupby('education')['y'].apply(lambda x: (x=='yes').mean()*100)
edu_sub.plot(kind='bar', ax=axes[1,2], color='teal', edgecolor='white', rot=0)
axes[1,2].set_title('Subscription Rate by Education (%)')
axes[1,2].set_ylabel('Subscription Rate %')

plt.tight_layout()
plt.savefig('eda_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Data Preprocessing

In [ ]:
# Label Encoding for categorical columns
le = LabelEncoder()
cat_cols = ['job','marital','education','default','housing','loan','contact','month','poutcome']

df_enc = df.copy()
for col in cat_cols:
    df_enc[col] = le.fit_transform(df_enc[col])

# Encode target
df_enc['y'] = (df_enc['y'] == 'yes').astype(int)

print("Encoding complete!")
print("\nSample encoded data:")
df_enc.head()

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(14, 10))
corr = df_enc.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, annot_kws={'size': 8})
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Train-Test Split
X = df_enc.drop('y', axis=1)
y = df_enc['y']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Training set size: {X_train.shape}")
print(f"Testing set size:  {X_test.shape}")
print(f"\nClass distribution in train set: {dict(y_train.value_counts())}")
print(f"Class distribution in test set:  {dict(y_test.value_counts())}")

## 5. Model Building — Decision Tree Classifier

In [ ]:
# Train the Decision Tree
clf = DecisionTreeClassifier(
    max_depth=5,
    criterion='gini',
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=42
)
clf.fit(X_train, y_train)
print("Model trained successfully!")
print(f"\nTree depth: {clf.get_depth()}")
print(f"Number of leaves: {clf.get_n_leaves()}")

## 6. Model Evaluation

In [ ]:
# Predictions
y_pred = clf.predict(X_test)

# Accuracy
acc = accuracy_score(y_test, y_pred)
print(f"✅ Model Accuracy: {acc:.4f} ({acc:.2%})")
print("\n" + "="*50)
print("Classification Report:")
print("="*50)
print(classification_report(y_test, y_pred, target_names=['No (0)', 'Yes (1)']))

In [ ]:
# Confusion Matrix + Feature Importance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No','Yes'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title(f'Confusion Matrix\nAccuracy: {acc:.2%}', fontsize=13)

# Feature Importances
feat_imp = pd.Series(clf.feature_importances_, index=X.columns).sort_values(ascending=True)
feat_imp.tail(10).plot(kind='barh', ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('Top 10 Feature Importances', fontsize=13)
axes[1].set_xlabel('Importance Score')

plt.tight_layout()
plt.savefig('model_results.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nTop 5 most important features:")
print(feat_imp.sort_values(ascending=False).head())

## 7. Decision Tree Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(24, 10))
plot_tree(clf, max_depth=3, feature_names=X.columns.tolist(),
          class_names=['No','Yes'], filled=True, rounded=True,
          fontsize=9, ax=ax, impurity=True)
ax.set_title('Decision Tree Visualization (max_depth=3 shown)', fontsize=15, fontweight='bold')
plt.savefig('decision_tree.png', dpi=120, bbox_inches='tight')
plt.show()
print("Decision tree saved!")

## 8. Hyperparameter Tuning — Depth Analysis

In [ ]:
# Test different max_depth values
depths = range(1, 15)
train_acc = []
test_acc = []

for d in depths:
    model = DecisionTreeClassifier(max_depth=d, random_state=42)
    model.fit(X_train, y_train)
    train_acc.append(accuracy_score(y_train, model.predict(X_train)))
    test_acc.append(accuracy_score(y_test, model.predict(X_test)))

plt.figure(figsize=(10, 5))
plt.plot(depths, train_acc, 'bo-', label='Train Accuracy', linewidth=2)
plt.plot(depths, test_acc, 'rs-', label='Test Accuracy', linewidth=2)
plt.axvline(x=5, color='green', linestyle='--', label='Chosen depth=5')
plt.xlabel('Max Depth')
plt.ylabel('Accuracy')
plt.title('Decision Tree Accuracy vs Max Depth')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('depth_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

best_depth = depths[test_acc.index(max(test_acc))]
print(f"Best test accuracy: {max(test_acc):.4f} at depth={best_depth}")

## 9. Conclusions & Key Insights

- **Call Duration** is the most important predictor — longer calls correlate strongly with subscription.
- **Previous campaign outcome** (poutcome=success) significantly increases subscription likelihood.
- **Account Balance** plays a role — customers with higher balances are more likely to subscribe.
- The model achieves solid accuracy on the test set.
- Class imbalance (No >> Yes) affects recall for the minority class — techniques like SMOTE or class weighting can improve this.

---
*Completed as part of Prodigy InfoTech Data Science Internship | Task 3*